# AIGC 5005 · Module 01 · Part 3
## Git and GitHub

Companion notebook to Part 3 of the Module 01 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 40 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module01-tool` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module01-tool python=3.11 ipykernel`, then `conda activate module01-tool`. If VS Code offers to install `ipykernel`, accept.

## 0. Set up a scratch repository

This notebook runs the real `git` commands from the module, in a throwaway folder called `scratch_part3`. Nothing touches your own projects, and nothing is pushed anywhere. The helper below runs a command and prints it the way a terminal would, so the output reads like a session.

It sets your name and email **only inside the scratch repository**, so your global Git settings are not changed.

In [ ]:
import os
import shutil
import stat
import subprocess
from pathlib import Path

if shutil.which("git") is None:
    raise SystemExit("git is not installed or not on your PATH. Install it from git-scm.com, then restart the kernel.")

SCRATCH = Path("scratch_part3").resolve()


def force_remove(folder):
    """Delete a folder, including the read-only files Git creates (needed on Windows)."""
    def make_writable(func, path, _exc):
        os.chmod(path, stat.S_IWRITE)
        func(path)
    shutil.rmtree(folder, onerror=make_writable)


if SCRATCH.exists():
    force_remove(SCRATCH)
SCRATCH.mkdir()
repo = SCRATCH / "module01-tool"
repo.mkdir()


def git(*args):
    """Run `git <args>` inside the scratch repository and print the command and its output."""
    result = subprocess.run(["git", *args], cwd=repo, capture_output=True, text=True)
    print("$ git", " ".join(args))
    print((result.stdout + result.stderr).rstrip() or "(no output)")
    print()


print("git is ready. Scratch repository:", repo)

## 1. The four places a change can live

| Where | What it holds |
|---|---|
| **Working directory** | The files as they are on your disk right now, including edits you have not recorded |
| **Staging area** | The changes you have selected for the next snapshot |
| **Local repository** | The committed history on your own machine |
| **Remote (GitHub)** | The copy your marker sees |

`git status` tells you where each change is **right now**. `git log` tells you what is **already in history**.

## 2. `git init`, then a new file: the working directory

`git init` starts tracking a folder. A file you have just created is in the working directory only, so Git lists it as *untracked*.

**Predict first:** what will `git status` say about `app.py`?

In [ ]:
git("init", "-b", "main")
git("config", "user.name", "Student Example")        # local to this scratch repository
git("config", "user.email", "student@example.com")

(repo / "app.py").write_text('print("Temperature Advisor")\n')
git("status", "--short")                              # ?? means untracked

## 3. `git add`: the staging area

`git add` moves a change into the staging area. The two-column output of `--short` shows staging on the left and working directory on the right, so `A ` means *added to the staging area, nothing different since*.

In [ ]:
git("add", "app.py")
git("status", "--short")

## 4. `git commit`: the local repository

A commit is a snapshot of whatever is staged. After it, `git status` has nothing to report and `git log` has one entry.

In [ ]:
git("commit", "-m", "Add temperature advisor skeleton")
git("status")
git("log", "--oneline")

## 5. Change a file: the three places at once

Edit `app.py`. The change exists in the working directory only, so `--short` shows a modification in the **right-hand** column. `git diff` shows exactly what changed.

In [ ]:
(repo / "app.py").write_text('print("Temperature Advisor")\nprint("Enter a temperature in Celsius")\n')
git("status", "--short")                              # " M": modified, not staged
git("diff")

Stage it and look again. The letter moves to the **left-hand** column, which means it is now staged. Then commit it.

In [ ]:
git("add", "app.py")
git("status", "--short")                              # "M ": staged
git("commit", "-m", "Prompt for a temperature")
git("log", "--oneline")

## 6. What makes a good commit

A commit should be one coherent change, and the message should say what changed and why. Marking looks at your history, so a repository with one commit called `update` is weaker than the same code arriving in several commits that show how you got there.

| Instead of | Write |
|---|---|
| `update` | `Add temperature range validation` |
| `fix` | `Handle non-numeric input without crashing` |
| `final version` | `Extract conversion logic into a function` |
| `asdf` | `Add README with setup and run instructions` |

## 7. Ignoring what should not be committed

A `venv` folder sits inside your project, so it can be committed by accident. Here we create a fake one with a few files in it, and watch Git offer to track it.

**Predict first:** what will `git status` list after we create `.venv`?

In [ ]:
(repo / ".venv" / "bin").mkdir(parents=True)
(repo / ".venv" / "bin" / "python").write_text("pretend this is a 50 MB interpreter\n")
git("status", "--short")

Now add a `.gitignore`. Git stops offering the folder, and the only new thing it lists is the `.gitignore` file itself. Commit that.

In [ ]:
(repo / ".gitignore").write_text(".venv/\nvenv/\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n.env\n.DS_Store\n")
git("status", "--short")
git("add", ".gitignore")
git("commit", "-m", "Add .gitignore")
git("ls-files")                                       # every file Git is tracking: no .venv

### Watch out: a `.gitignore` only affects files added in future

If `.venv` had already been committed, adding the line would not remove it. You would run `git rm -r --cached .venv` (which stops tracking it without deleting the folder) and commit again. That is why the `.gitignore` is the **first** file you create in a new project.

## 8. Connecting to GitHub

Everything so far was local. A remote is a second copy somewhere else. This repository has none, so `git remote -v` is empty. In your own project, after creating an empty repository on GitHub, you would run the lines below (not here: there is nothing to push to).

```
git branch -M main
git remote add origin https://github.com/YOUR-USERNAME/module01-tool.git
git push -u origin main
```

In [ ]:
git("remote", "-v")                                   # nothing yet
git("log", "--oneline")                                # what a push would send

---
## Try it

In your real `module01-tool` folder (not here), repeat this sequence: create `.gitignore` first, `git init`, `git status`, make a README, stage, commit, create an empty GitHub repository and push. Then open the repository in your browser and confirm no environment folder is there.

## Clean up

Removes the scratch repository. `force_remove` handles the read-only files Git keeps in `.git`.

In [ ]:
force_remove(SCRATCH)
print("Scratch folder removed:", not SCRATCH.exists())